# 05 Candidate Model Training

This notebook trains lightweight YOLO candidate architectures for sign-detection architecture triage.


## Purpose

Notebook 05 compares candidate architectures only. Every candidate trains on `data_exp_D_full_pipeline.yaml`, which represents the strongest training condition: original train data plus offline augmented train data, with online augmentation enabled during training.

The untouched test split is not used here. Candidate selection is based on validation metrics and validation-image latency only.

To keep the repository readable, this notebook writes only the essential training reports by default. Per-model Ultralytics plots and notebook summary figures are disabled unless you explicitly turn them on.


## Step 1 - Setup

This cell locates the `sign-detection` module root, adds it to `sys.path`, and sets a few Windows/Jupyter environment defaults before importing Ultralytics-facing helpers. The environment settings reduce common OpenMP/MKL conflicts when torch starts inside a notebook kernel.


In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

# Keep Windows/Jupyter torch startup stable before Ultralytics is imported.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import matplotlib.pyplot as plt  # Used only when optional summary figures are enabled.
import pandas as pd
import yaml
from IPython.display import display


def find_project_root(start: Path) -> Path:
    """Walk upward until the sign-detection project root is found."""
    for candidate in [start, *start.parents]:
        if (
            candidate / "configs" / "training_config.yaml"
        ).exists() and candidate.name == "sign-detection":
            return candidate
        if (candidate / "sign-detection" / "configs" / "training_config.yaml").exists():
            return candidate / "sign-detection"
    raise FileNotFoundError("Could not locate the sign-detection project root.")


PROJECT_ROOT = find_project_root(Path.cwd()).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.training.evaluate_yolo import (  # noqa: E402
    benchmark_model_latency,
    evaluate_candidate_model,
    rank_candidate_results,
)
from src.training.train_yolo import train_candidate_models  # noqa: E402

print(f"Project root: {PROJECT_ROOT}")


Project root: C:\Github\smart-factory-safety-monitoring-system\sign-detection


## Step 2 - Load Configuration

The candidate list and training settings come from `configs/training_config.yaml`. Online augmentation settings come from `configs/augmentation_config.yaml` and are passed as training arguments, not encoded into the dataset YAML.


In [2]:
def load_yaml(path: Path) -> dict:
    """Load one YAML file using UTF-8 encoding."""
    with path.open("r", encoding="utf-8") as file_handle:
        return yaml.safe_load(file_handle)


training_config = load_yaml(PROJECT_ROOT / "configs" / "training_config.yaml")
augmentation_config = load_yaml(PROJECT_ROOT / "configs" / "augmentation_config.yaml")
class_config = load_yaml(PROJECT_ROOT / "configs" / "class_names.yaml")

candidate_models = training_config.get("candidate_models", [])
if not candidate_models:
    raise ValueError(
        "No candidate models are configured in configs/training_config.yaml"
    )

online_aug_args = dict(augmentation_config.get("online_augmentation_on", {}))
if not online_aug_args:
    raise ValueError(
        "online_augmentation_on is missing from configs/augmentation_config.yaml"
    )

print(f"Loaded {len(candidate_models)} candidate models.")
print(f"Classes: {class_config.get('names')}")


Loaded 6 candidate models.
Classes: {0: 'M014_Helmet', 1: 'M015_Vest', 2: 'P004_NoThoroughfare', 3: 'W011_Slippery'}


## Step 3 - Verify Dataset YAML

Candidate triage uses only `data_exp_D_full_pipeline.yaml`. The notebook checks that the dataset YAML exists and that the train and validation image folders referenced by it are present. The test path may exist in the YAML for final use later, but it is not used for ranking candidates here.


In [3]:
data_yaml = PROJECT_ROOT / "data_exp_D_full_pipeline.yaml"
if not data_yaml.exists():
    raise FileNotFoundError(
        "Missing data_exp_D_full_pipeline.yaml. Run Notebook 04 before candidate model training."
    )

data_config = load_yaml(data_yaml)
dataset_root = (PROJECT_ROOT / data_config["path"]).resolve()
train_images_dir = dataset_root / data_config["train"]
val_images_dir = dataset_root / data_config["val"]
test_images_dir = dataset_root / data_config.get("test", "test/images")

required_dirs = {
    "dataset_root": dataset_root,
    "train_images": train_images_dir,
    "val_images": val_images_dir,
}
missing_dirs = {name: path for name, path in required_dirs.items() if not path.exists()}
if missing_dirs:
    raise FileNotFoundError(f"Required dataset folders are missing: {missing_dirs}")

print(f"Dataset YAML: {data_yaml.relative_to(PROJECT_ROOT)}")
print(f"Train images: {train_images_dir}")
print(f"Val images: {val_images_dir}")
print(f"Test images listed for later final evaluation only: {test_images_dir}")


Dataset YAML: data_exp_D_full_pipeline.yaml
Train images: C:\Github\smart-factory-safety-monitoring-system\sign-detection\data\generated\experiments\exp_D_full_pipeline\train\images
Val images: C:\Github\smart-factory-safety-monitoring-system\sign-detection\data\generated\experiments\exp_D_full_pipeline\val\images
Test images listed for later final evaluation only: C:\Github\smart-factory-safety-monitoring-system\sign-detection\data\generated\experiments\exp_D_full_pipeline\test\images


## Step 4 - Review Candidate Models

This table is the exact candidate list that will be attempted. If a checkpoint is unsupported or unavailable in the installed Ultralytics version, it will be marked as failed while the remaining candidates continue.


In [4]:
candidate_table = pd.DataFrame(candidate_models)
display(candidate_table)


,name,weights
0,YOLOv8_Nano,yolov8n.pt
1,YOLOv9_Tiny,yolov9t.pt
2,YOLOv10_Nano,yolov10n.pt
3,YOLO11_Nano,yolo11n.pt
4,YOLO12_Nano,yolo12n.pt
5,YOLO26_Nano,yolo26n.pt


## Step 5 - Build Shared Training Arguments

All candidates use the same epochs, image size, batch size, patience, device, worker count, seed, and online augmentation values. `DRY_RUN` can be set to `True` to verify orchestration without starting training.

Artifact control is handled here too: `SAVE_ULTRALYTICS_PLOTS=False` avoids creating multiple PNG plots inside every candidate run, and `SAVE_SUMMARY_FIGURES=False` avoids creating notebook-level chart files. The ranking CSVs remain the main record of the trial.


In [5]:
# Set DRY_RUN=True when checking notebook wiring without launching training.
DRY_RUN = False

# Keep existing candidate runs unless you intentionally want to replace them.
OVERWRITE_EXISTING_RUNS = False

# Training already evaluates on val. Leave this off unless you need a fresh best.pt validation pass.
RUN_POST_TRAIN_VALIDATION = False

# Latency is a compact scalar metric, so it is useful for ranking and does not create extra files.
RUN_LATENCY_BENCHMARK = True
MAX_BENCHMARK_IMAGES = 20

# Artifact controls. Keep these off for normal experimentation to avoid many per-run images.
SAVE_ULTRALYTICS_PLOTS = False
SAVE_SUMMARY_FIGURES = False

triage_epochs = int(
    training_config.get("candidate_epochs", training_config.get("epochs", 50))
)
train_args = {
    "epochs": triage_epochs,
    "imgsz": int(training_config.get("imgsz", 640)),
    "batch": int(training_config.get("batch", 16)),
    "device": training_config.get("device", 0),
    "workers": int(training_config.get("workers", 8)),
    "patience": int(training_config.get("patience", 30)),
    "seed": int(training_config.get("seed", 42)),
    "plots": SAVE_ULTRALYTICS_PLOTS,
    "dry_run": DRY_RUN,
    "overwrite": OVERWRITE_EXISTING_RUNS,
}

# Optional config key: if absent, let Ultralytics choose its default AMP behavior.
if "amp" in training_config:
    train_args["amp"] = bool(training_config["amp"])

# Online augmentation is intentionally ON for exp_D candidate triage.
# These values affect training only; the dataset YAML stays clean and reusable.
train_args.update(online_aug_args)

training_settings_preview = pd.DataFrame(
    [{"setting": key, "value": value} for key, value in train_args.items()]
)
display(training_settings_preview)


,setting,value
0,epochs,50
1,imgsz,640
2,batch,16
3,device,0
4,workers,8
5,patience,30
6,seed,42
7,plots,False
8,dry_run,False
9,overwrite,False


## Step 6 - Train Candidate Models

This is the training cell. It writes runs under `runs/candidate_models/`. When `OVERWRITE_EXISTING_RUNS=False`, existing runs are preserved and a numeric suffix is added to the new run name.


In [6]:
candidate_runs_dir = PROJECT_ROOT / "runs" / "candidate_models"
reports_training_dir = PROJECT_ROOT / "reports" / "training"
reports_training_dir.mkdir(parents=True, exist_ok=True)

# The helper handles unsupported weights safely and returns one row per candidate.
# With plots disabled, each run mainly contains weights/logs instead of a pile of images.

candidate_results = train_candidate_models(
    data_yaml=data_yaml,
    candidate_models=candidate_models,
    output_dir=candidate_runs_dir,
    train_args=train_args,
    continue_on_error=True,
)

display(candidate_results)


New https://pypi.org/project/ultralytics/8.4.66 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.51  Python-3.10.20 torch-2.13.0.dev20260517+cu132 CUDA:0 (NVIDIA GeForce RTX 5060 Ti, 16311MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Github\smart-factory-safety-monitoring-system\sign-detection\runs\candidate_models\_data_exp_D_full_pipeline_resolved.yaml, degrees=10, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.4, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01,

,model_name,weights,run_name,status,data_yaml,resolved_data_yaml,run_dir,precision,recall,map50,...,fitness,training_time,best_weights_path,last_weights_path,model_size_mb,fps,avg_latency_ms,num_benchmark_images,error_message,notes
0,YOLOv8_Nano,yolov8n.pt,YOLOv8_Nano_expD,trained,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,0.95241,0.95115,0.96592,...,None,210.586,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,5.961222,None,None,None,,
1,YOLOv9_Tiny,yolov9t.pt,YOLOv9_Tiny_expD,trained,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,0.94534,0.98154,0.98399,...,None,364.682,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,4.424081,None,None,None,,
2,YOLOv10_Nano,yolov10n.pt,YOLOv10_Nano_expD,trained,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,0.92351,0.92359,0.94865,...,None,281.365,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,5.487665,None,None,None,,
3,YOLO11_Nano,yolo11n.pt,YOLO11_Nano_expD,trained,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,0.94338,0.96181,0.97753,...,None,229.108,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,5.219690,None,None,None,,
4,YOLO12_Nano,yolo12n.pt,YOLO12_Nano_expD,trained,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,0.96137,0.94115,0.96820,...,None,308.119,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,5.263880,None,None,None,,
5,YOLO26_Nano,yolo26n.pt,YOLO26_Nano_expD,trained,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,0.90320,0.89395,0.93390,...,None,286.928,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,5.139531,None,None,None,,


## Step 7 - Optional Post-Training Validation and Latency

Training already validates each candidate on the validation split. This cell can optionally run an extra validation pass from `best.pt`, then benchmarks inference speed on a small deterministic subset of validation images. The test split is still not used.


In [7]:
updated_rows = []
for row in candidate_results.to_dict("records"):
    # Failed or skipped candidates stay in the report but are not benchmarked.
    if row.get("status") != "trained":
        updated_rows.append(row)
        continue

    best_weights_path = Path(str(row.get("best_weights_path", "")))
    if RUN_POST_TRAIN_VALIDATION and best_weights_path.exists():
        eval_metrics = evaluate_candidate_model(
            weights_path=best_weights_path,
            data_yaml=data_yaml,
            imgsz=int(train_args["imgsz"]),
            device=train_args["device"],
        )
        # Keep training status as trained; only replace metric values when validation succeeded.
        if eval_metrics.get("status") == "evaluated":
            for metric_name in ["precision", "recall", "map50", "map50_95", "fitness"]:
                row[metric_name] = eval_metrics.get(metric_name, row.get(metric_name))
        else:
            row["notes"] = (
                f"post-training validation warning: {eval_metrics.get('error_message', '')}"
            )

    # Benchmark on validation images only; never use the untouched test split for triage.
    if RUN_LATENCY_BENCHMARK and best_weights_path.exists():
        latency_metrics = benchmark_model_latency(
            weights_path=best_weights_path,
            sample_images_dir=val_images_dir,
            imgsz=int(train_args["imgsz"]),
            device=train_args["device"],
            max_images=MAX_BENCHMARK_IMAGES,
        )
        row.update(latency_metrics)

    updated_rows.append(row)

candidate_results = pd.DataFrame(updated_rows)
display(candidate_results)


,model_name,weights,run_name,status,data_yaml,resolved_data_yaml,run_dir,precision,recall,map50,...,training_time,best_weights_path,last_weights_path,model_size_mb,fps,avg_latency_ms,num_benchmark_images,error_message,notes,latency_warning
0,YOLOv8_Nano,yolov8n.pt,YOLOv8_Nano_expD,trained,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,0.95241,0.95115,0.96592,...,210.586,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,5.961222,27.025566,37.002000,20,,,
1,YOLOv9_Tiny,yolov9t.pt,YOLOv9_Tiny_expD,trained,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,0.94534,0.98154,0.98399,...,364.682,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,4.424081,23.358316,42.811305,20,,,
2,YOLOv10_Nano,yolov10n.pt,YOLOv10_Nano_expD,trained,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,0.92351,0.92359,0.94865,...,281.365,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,5.487665,28.146346,35.528590,20,,,
3,YOLO11_Nano,yolo11n.pt,YOLO11_Nano_expD,trained,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,0.94338,0.96181,0.97753,...,229.108,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,5.219690,27.528299,36.326255,20,,,
4,YOLO12_Nano,yolo12n.pt,YOLO12_Nano_expD,trained,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,0.96137,0.94115,0.96820,...,308.119,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,5.263880,25.937666,38.553970,20,,,
5,YOLO26_Nano,yolo26n.pt,YOLO26_Nano_expD,trained,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,0.90320,0.89395,0.93390,...,286.928,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,5.139531,27.255970,36.689210,20,,,


## Step 8 - Rank Candidates

Successful candidates are sorted by the project priority: recall first, then mAP50, mAP50-95, FPS, and finally smaller model size. This ranking uses validation metrics only.


In [8]:
candidate_ranking = rank_candidate_results(candidate_results)

results_path = reports_training_dir / "candidate_model_results.csv"
failures_path = reports_training_dir / "candidate_model_failures.csv"
ranking_path = reports_training_dir / "candidate_model_ranking.csv"

# Keep the persistent output compact: results, failures, and ranking CSVs only.
candidate_results.to_csv(results_path, index=False)

failure_columns = ["model_name", "weights", "status", "error_message"]
failures_df = candidate_results.loc[
    candidate_results["status"].eq("failed"), failure_columns
].copy()
# Always write a failures CSV, even when there are no failures, so downstream notebooks have a stable contract.
if failures_df.empty:
    failures_df = pd.DataFrame(columns=failure_columns)
failures_df.to_csv(failures_path, index=False)

candidate_ranking.to_csv(ranking_path, index=False)

print(f"Saved results: {results_path.relative_to(PROJECT_ROOT)}")
print(f"Saved failures: {failures_path.relative_to(PROJECT_ROOT)}")
print(f"Saved ranking: {ranking_path.relative_to(PROJECT_ROOT)}")

display(candidate_ranking)


Saved results: reports\training\candidate_model_results.csv
Saved failures: reports\training\candidate_model_failures.csv
Saved ranking: reports\training\candidate_model_ranking.csv


,rank,model_name,weights,run_name,status,data_yaml,resolved_data_yaml,run_dir,precision,recall,...,training_time,best_weights_path,last_weights_path,model_size_mb,fps,avg_latency_ms,num_benchmark_images,error_message,notes,latency_warning
0,1,YOLOv9_Tiny,yolov9t.pt,YOLOv9_Tiny_expD,trained,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,0.94534,0.98154,...,364.682,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,4.424081,23.358316,42.811305,20,,,
1,2,YOLO11_Nano,yolo11n.pt,YOLO11_Nano_expD,trained,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,0.94338,0.96181,...,229.108,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,5.219690,27.528299,36.326255,20,,,
2,3,YOLOv8_Nano,yolov8n.pt,YOLOv8_Nano_expD,trained,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,0.95241,0.95115,...,210.586,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,5.961222,27.025566,37.002000,20,,,
3,4,YOLO12_Nano,yolo12n.pt,YOLO12_Nano_expD,trained,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,0.96137,0.94115,...,308.119,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,5.263880,25.937666,38.553970,20,,,
4,5,YOLOv10_Nano,yolov10n.pt,YOLOv10_Nano_expD,trained,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,0.92351,0.92359,...,281.365,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,5.487665,28.146346,35.528590,20,,,
5,6,YOLO26_Nano,yolo26n.pt,YOLO26_Nano_expD,trained,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,0.90320,0.89395,...,286.928,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,5.139531,27.255970,36.689210,20,,,


## Step 9 - Optional Summary Figures

Summary figures are disabled by default to avoid cluttering `reports/figures/` after every trial. Turn `SAVE_SUMMARY_FIGURES=True` in Step 5 only when you need a few quick visuals for a report.


In [9]:
if not SAVE_SUMMARY_FIGURES:
    print(
        "Summary figures are disabled. Set SAVE_SUMMARY_FIGURES=True in Step 5 if you need them."
    )
else:
    figures_dir = PROJECT_ROOT / "reports" / "figures"
    figures_dir.mkdir(parents=True, exist_ok=True)

    successful = candidate_ranking.copy()
    if successful.empty:
        print("No successful candidate rows available for plotting.")
    else:
        plot_specs = [
            ("recall", "Validation Recall", figures_dir / "candidate_model_recall.png"),
            ("map50", "Validation mAP50", figures_dir / "candidate_model_map50.png"),
            (
                "avg_latency_ms",
                "Average Latency (ms)",
                figures_dir / "candidate_model_latency.png",
            ),
        ]
        for column, title, output_path in plot_specs:
            if column not in successful.columns or successful[column].dropna().empty:
                print(f"Skipped {title}: no {column} values available.")
                continue
            plt.figure(figsize=(8, 4))
            plt.bar(successful["model_name"], successful[column])
            plt.title(title)
            plt.xlabel("Candidate")
            plt.ylabel(column)
            plt.xticks(rotation=30, ha="right")
            plt.tight_layout()
            plt.savefig(output_path, dpi=150)
            plt.show()
            print(f"Saved figure: {output_path.relative_to(PROJECT_ROOT)}")


Summary figures are disabled. Set SAVE_SUMMARY_FIGURES=True in Step 5 if you need them.


## Step 10 - Notebook 06 Checklist

Before moving to Notebook 06:

- Candidate training results were saved to `reports/training/candidate_model_results.csv`.
- Failed or unsupported models were reviewed in `candidate_model_failures.csv`.
- The ranking was saved to `candidate_model_ranking.csv`.
- Per-run plots stayed disabled unless `SAVE_ULTRALYTICS_PLOTS=True` was intentionally set.
- Optional summary figures stayed disabled unless `SAVE_SUMMARY_FIGURES=True` was intentionally set.
- The selected architecture is based on validation metrics, not test metrics.
- Notebook 06 can use the top successful candidate for ablation training.
